# Implementing a custom interface: `ase.io.read`

This tutorial shows how to create a lightweight Moliterate interface that reads structures with `ase.io.read`.
We'll build a minimal dataset class that wraps `ase.io.read`, exposes `ChemDataEntry` rows, and integrates with Moliterate transforms.


<div class="alert alert-block alert-warning">
<b>Pre-requisites:</b> 
Before starting this tutorial be sure to know the basics explained in:

- __[quick-overview.ipynb](quick-overview.ipynb)__

- __[core-moliterate-concepts.ipynb](core-moliterate-concepts.ipynb)__
</div>

## Setup
We'll create a tiny `extxyz` file on the fly so the example is fully self-contained.


In [1]:
from __future__ import annotations

import tempfile
from pathlib import Path
from typing import Union

import numpy as np
from ase.build import molecule
from ase.io import write

from scm.moliterate.core.base_chem_dataset import BaseChemDataSet


## Create a small input file
We store simple metadata (`energy`) and per-atom arrays (`forces`) to demonstrate how properties can be extracted.


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix="moliterate-ase-io-"))
xyz_path = tmp_dir / "demo.extxyz"

atoms1 = molecule("H2O")
atoms1.info["energy"] = -76.0
atoms1.arrays["forces"] = np.zeros((len(atoms1), 3))

atoms2 = molecule("NH3")
atoms2.info["energy"] = -56.0
atoms2.arrays["forces"] = np.zeros((len(atoms2), 3))

write(xyz_path, [atoms1, atoms2], format="extxyz")
xyz_path


PosixPath('/tmp/moliterate-ase-io-k65cqcwj/demo.extxyz')

In [3]:
# Visualize the file
with open(xyz_path) as f:
    print(f.read())

3
Properties=species:S:1:pos:R:3:forces:R:3 energy=-76.0 pbc="F F F"
O        0.00000000       0.00000000       0.11926200       0.00000000       0.00000000       0.00000000
H        0.00000000       0.76323900      -0.47704700       0.00000000       0.00000000       0.00000000
H        0.00000000      -0.76323900      -0.47704700       0.00000000       0.00000000       0.00000000
4
Properties=species:S:1:pos:R:3:forces:R:3 energy=-56.0 pbc="F F F"
N        0.00000000       0.00000000       0.11648900       0.00000000       0.00000000       0.00000000
H        0.00000000       0.93973100      -0.27180800       0.00000000       0.00000000       0.00000000
H        0.81383100      -0.46986500      -0.27180800       0.00000000       0.00000000       0.00000000
H       -0.81383100      -0.46986500      -0.27180800       0.00000000       0.00000000       0.00000000



## The minimal interface
A custom Moliterate interface is a subclass of `BaseChemDataSet` that implements:
- `total_len` and `metadata`

- `__iter__` and `get_row`

- `available_properties`

We'll keep this interface intentionally small but still support transforms and property extraction.


### Implementation

First step implement a class that contains the data path of the file:

In [4]:
class DSPath(BaseChemDataSet):
    """Moliterate interface that reads structures using `ase.io.read`."""

    data_source: Union[str, Path]

#### `total_len` and `metadata`
First step is the `total_len` and `metadata`:

In [5]:
from typing import Any, Dict


def total_len(self) -> int:
    """method to get the total len of the data"""
    ...

@property
def metadata(self) -> Dict[str, Any]:
    """Global metadata"""
    ...

A possible solution could be:

In [6]:
from typing import Any, List, Optional

from ase.io import read
from pydantic import PrivateAttr


class LenDSPath(DSPath):
    """Moliterate interface that reads structures using `ase.io.read`."""

    format: Optional[str] = None

    _atoms_cache: Optional[List[Any]] = PrivateAttr(default=None)

    def _read_atoms(self) -> List[Any]:
        if self._atoms_cache is None:
            atoms = read(self.data_source, index=":", format=self.format)
            if isinstance(atoms, list):
                self._atoms_cache = atoms
            else:
                self._atoms_cache = [atoms]
        return list(self._atoms_cache)
    
    def total_len(self) -> int:
        return len(self._read_atoms())


#### `__iter__` and `get_row`

In [7]:
from typing import Iterator

from scm.moliterate.core.chem_data_entry import ChemDataEntry


def __iter__(self) -> Iterator[ChemDataEntry]: ...  # pyright: ignore[reportIncompatibleMethodOverride]

def get_row(self, idx: int) -> ChemDataEntry: ...

In [8]:
from typing import Dict


class ReadLenDSPath(LenDSPath):
    """Moliterate interface that reads structures using `ase.io.read`."""

    format: Optional[str] = None

    _atoms_cache: Optional[List[Any]] = PrivateAttr(default=None)

    def _read_atoms(self) -> List[Any]:
        if self._atoms_cache is None:
            atoms = read(self.data_source, index=":", format=self.format)
            if isinstance(atoms, list):
                self._atoms_cache = atoms
            else:
                self._atoms_cache = [atoms]
        return list(self._atoms_cache)

    def get_row(self, idx: int) -> ChemDataEntry:
        # from relative idx to absolute idx
        idx_absolute = int(self.select_indices(indices=idx)[0])
        atoms = self._read_atoms()[idx_absolute]
        row = self._to_chem_data_entry(idx_absolute, atoms)
        return self.composed_transform(row)

    
    def _to_chem_data_entry(self, idx_absolute, atoms):
        row = ChemDataEntry(
                system=atoms.copy(),
                properties=self._extract_properties(atoms),
                idx_absolute=idx_absolute,
                idx_origin=idx_absolute,
            )
        return row
    
    def _extract_properties(self, atoms) -> Dict[str, Any]:
        properties: Dict[str, Any] = {}
        properties.update(atoms.info)
        for k in atoms.arrays:
            if k not in ["positions", "numbers"]:
                properties[k] = atoms.arrays[k]
        if atoms.calc is not None:
            properties.update(atoms.calc.results)
        return properties

    def __iter__(self):
        transform = self.composed_transform
        atoms_list = self._read_atoms()
        for idx_absolute in map(int, self.select_indices()):
            atoms = atoms_list[idx_absolute]
            row = self._to_chem_data_entry(idx_absolute, atoms)
            yield transform(row)

#### `available_properties`

In [9]:
from scm.moliterate.core.properties_info import PropertyInfo


@property
def available_properties(self) -> List[PropertyInfo]:
    """Available properties in the dataset"""
    ...

In [10]:
class PropReadLenDSPath(ReadLenDSPath):

    _properties_cache: Optional[List[PropertyInfo]] = PrivateAttr(default=None)

    @property
    def available_properties(self) -> List[PropertyInfo]:
        if self._properties_cache is None:
            ret = []
            for entry_i in self:
                for p in entry_i.properties.keys():
                    ret.append(PropertyInfo(name=p))
            self._properties_cache = list(set(ret))
        return self._properties_cache

#### Nice to have

It is nice to add some validation, since the data_source is required for our interface to be working:

In [11]:
from pydantic import field_validator


class ValidPropReadLenDSPath(PropReadLenDSPath):
    """Moliterate interface that reads structures using `ase.io.read`."""

    data_source: Union[str, Path]

    @field_validator("data_source", mode="before")
    @classmethod
    def _validate_data_source(cls, value: Any) -> str:
        path = str(value)
        if not Path(path).exists():
            raise FileNotFoundError(f"Data source not found: {path}")
        return path

Important to add as well the type field, which is later used as model discriminator in [pydantic](https://docs.pydantic.dev/latest/concepts/unions/).

In [12]:
from typing import Literal


class ASEReadMolData(ValidPropReadLenDSPath):
    """Moliterate interface that reads structures using `ase.io.read`."""

    type: Literal["ase_read"] = "ase_read"
    


### Final Result

In [13]:
class ASEReadMolData(BaseChemDataSet):
    """Moliterate interface that reads structures using `ase.io.read`."""

    type: Literal["ase_read"] = "ase_read"
    data_source: Union[str, Path]
    format: Optional[str] = None

    _atoms_cache: Optional[List[Any]] = PrivateAttr(default=None)
    _properties_cache: Optional[List[PropertyInfo]] = PrivateAttr(default=None)

    @field_validator("data_source", mode="before")
    @classmethod
    def _validate_data_source(cls, value: Any) -> str:
        path = str(value)
        if not Path(path).exists():
            raise FileNotFoundError(f"Data source not found: {path}")
        return path

    def _read_atoms(self) -> List[Any]:
        if self._atoms_cache is None:
            atoms = read(self.data_source, index=":", format=self.format)
            if isinstance(atoms, list):
                self._atoms_cache = atoms
            else:
                self._atoms_cache = [atoms]
        return list(self._atoms_cache)
    
    def total_len(self) -> int:
        return len(self._read_atoms())

    @property
    def available_properties(self) -> List[PropertyInfo]:
        if self._properties_cache is None:
            ret = []
            for entry_i in self[:5]:
                for p in entry_i.properties.keys():
                    ret.append(PropertyInfo(name=p))
            self._properties_cache = list(set(ret))
        return self._properties_cache


    @property
    def metadata(self) -> Dict[str, Any]:
        return {}

    def __iter__(self):
        transform = self.composed_transform
        atoms_list = self._read_atoms()
        for idx_absolute in map(int, self.select_indices()):
            atoms = atoms_list[idx_absolute]
            row = self._to_chem_data_entry(idx_absolute, atoms)
            yield transform(row)

    def get_row(self, idx: int) -> ChemDataEntry:
        idx_absolute = int(self.select_indices(indices=idx)[0])
        atoms = self._read_atoms()[idx_absolute]
        row = self._to_chem_data_entry(idx_absolute, atoms)
        return self.composed_transform(row)

    def _to_chem_data_entry(self, idx_absolute, atoms):
        row = ChemDataEntry(
                system=atoms.copy(),
                properties=self._extract_properties(atoms),
                idx_absolute=idx_absolute,
                idx_origin=idx_absolute,
            )
        return row
    
    def _extract_properties(self, atoms) -> Dict[str, Any]:
        properties: Dict[str, Any] = {}
        properties.update(atoms.info)
        for k in atoms.arrays:
            if k not in ["positions", "numbers"]:
                properties[k] = atoms.arrays[k]
        if atoms.calc is not None:
            properties.update(atoms.calc.results)
        return properties


## Use the interface
Instantiate the dataset and inspect rows just like any other Moliterate dataset.


In [14]:
ds = ASEReadMolData(
    data_source=xyz_path,
    format="extxyz",
)

ds

ASEReadMolData(len=2, properties=['energy', 'forces'], transforms=[] at 0x7ccd00c894a0)

In [15]:
ds.available_properties


[PropertyInfo(name='energy', unit=None, shape=None, description=''),
 PropertyInfo(name='forces', unit=None, shape=None, description='')]

In [16]:
ds[0].properties

{'forces': array([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]),
 'energy': -76.0}

## Next steps

- Add the implementation in a file inside `/src/scm/moliterate/interfaces` and add the class in the `UnionInterfaces`.

- Add this extra implementation to `load_dataset` to auto-detect this interface, register it in `scm.moliterate.core.interface`.

- Tests: in the config.py, add the paths in `ITER_PATHS_AVAIL` for the extra implementations and add the data in `/tutorials/data`.

- Extend the `ASEReadMolData` to be a `BaseChemDataSetWriter` sitting on top of `ase.io.write`.